In [7]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "heilbronner2008fruit")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "heilbronner_etal_2008_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [8]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="heilbronner2008fruit"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"subject": "ape",
    "species": "species_original"})


In [9]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [10]:
# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

In [11]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True) 

In [12]:

heilbronner2008fruit_standardized=df[['study_id','participant',  'age_in_years','sex','species', 'session', 'trial','trial_type',
       'safe_choice', 'side']]
comp_out_path_stand = os.path.join(out_pathway, 'heilbronner2008fruit_standardized.csv')
heilbronner2008fruit_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [13]:

names= heilbronner2008fruit_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
heilbronner2008fruit_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'heilbronner2008fruit_glossary.csv')
heilbronner2008fruit_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
